# DESeq2 Reanalysis Validation

Validate DESeq2 reanalysis results against published data using **official NCBI gene ID mapping**.

## The Problem

Publications may use different genome annotation versions than your reanalysis:
- **V2** (GCA_002759435.2): 6-digit gene IDs (e.g., `B9J08_001458`)
- **V3** (GCA_002759435.3): 5-digit gene IDs (e.g., `B9J08_03708`)

## Why NOT to Use LFC Matching

A "clever" approach is to match genes by log2 fold change similarity. **Don't do this.**

LFC matching achieved R² = 0.9996 but was **99% wrong**—it found genes with coincidentally similar fold changes, not the actual corresponding genes.

## Correct Approach

1. **GTF `old_locus_tag`**: NCBI's V3 GTF contains official mapping to V2 gene IDs
2. **Protein validation**: Confirm mapping by comparing protein sequences (should be identical)

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*narwhals.*')

import pandas as pd
import numpy as np
from scipy import stats
import altair as alt
import re
from pathlib import Path

alt.data_transformers.enable('default', max_rows=None)

## Configuration

In [ ]:
# === GALAXY JUPYTERLITE: Use history dataset numbers ===
# Set these to Galaxy history IDs when running in JupyterLite
GTF_V3_HISTORY_ID = None       # V3 GTF with old_locus_tag
PROTEIN_V2_HISTORY_ID = None   # V2 protein FASTA
PROTEIN_V3_HISTORY_ID = None   # V3 protein FASTA
DESEQ2_HISTORY_ID = None       # DESeq2 results TSV
PUBLICATION_HISTORY_ID = None  # Publication DEGs CSV

# === LOCAL JUPYTER: Use file paths ===
# Set these when running locally (history IDs should be None)
GTF_V3_FILE = "../shared_reference/GCA_002759435.3_Cand_auris_B8441_V3_genomic.gtf"
PROTEIN_V2_FILE = "../shared_reference/GCA_002759435.2_Cand_auris_B8441_V2_protein.faa"
PROTEIN_V3_FILE = "../shared_reference/GCA_002759435.3_Cand_auris_B8441_V3_protein.faa"
DESEQ2_FILE = "../santana24_PRJNA904261/analysis/deseq2_tnSWI1.tsv"
PUBLICATION_FILE = "santana24_publication_data.csv"

# === COMPARISON INFO ===
COMPARISON_NAME = "Santana et al. (2024) tnSWI1"

# Key genes to label on plots (v2_id: name)
KEY_GENES = {
    'B9J08_001458': 'SCF1',
    'B9J08_000656': 'MGD1',
}

## Gene ID Mapping Functions

In [ ]:
def parse_gtf_gene_mapping(gtf_path):
    """
    Extract v2→v3 gene ID mapping from GTF old_locus_tag attribute.
    
    Returns:
        dict: {v2_gene_id: v3_gene_id}
    """
    v2_to_v3 = {}
    with open(gtf_path) as f:
        for line in f:
            if line.startswith('#'):
                continue
            if '\tgene\t' not in line:
                continue
            
            gene_match = re.search(r'gene_id "([^"]+)"', line)
            old_match = re.search(r'old_locus_tag "([^"]+)"', line)
            
            if gene_match and old_match:
                v3_id = gene_match.group(1)
                v2_id = old_match.group(1)
                v2_to_v3[v2_id] = v3_id
    
    print(f"Loaded {len(v2_to_v3)} gene ID mappings from GTF")
    return v2_to_v3

In [ ]:
def parse_fasta(fasta_path):
    """
    Parse FASTA file to extract gene_id → protein_sequence mapping.
    
    Returns:
        dict: {gene_id: sequence}
    """
    sequences = {}
    current_gene = None
    current_seq = []
    
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                # Save previous sequence
                if current_gene and current_seq:
                    sequences[current_gene] = ''.join(current_seq)
                
                # Extract gene ID (B9J08_XXXXX pattern)
                gene_match = re.search(r'(B9J08_\d+)', line)
                current_gene = gene_match.group(1) if gene_match else None
                current_seq = []
            else:
                if current_gene:
                    current_seq.append(line)
        
        # Save last sequence
        if current_gene and current_seq:
            sequences[current_gene] = ''.join(current_seq)
    
    print(f"Loaded {len(sequences)} protein sequences from {Path(fasta_path).name}")
    return sequences

In [ ]:
def validate_mapping_with_proteins(gene_mapping, v2_seqs, v3_seqs):
    """
    Validate gene ID mapping by comparing protein sequences.
    
    Returns:
        DataFrame with validation results
    """
    results = []
    
    for v2_id, v3_id in gene_mapping.items():
        v2_seq = v2_seqs.get(v2_id)
        v3_seq = v3_seqs.get(v3_id)
        
        if v2_seq and v3_seq:
            exact_match = v2_seq == v3_seq
            identity = 100.0 if exact_match else None
        else:
            exact_match = False
            identity = None
        
        results.append({
            'v2_gene': v2_id,
            'v3_gene': v3_id,
            'v2_length': len(v2_seq) if v2_seq else None,
            'v3_length': len(v3_seq) if v3_seq else None,
            'exact_match': exact_match,
            'identity': identity
        })
    
    df = pd.DataFrame(results)
    
    # Summary stats
    compared = df['exact_match'].notna().sum()
    matched = df['exact_match'].sum()
    print(f"Protein validation: {matched}/{compared} exact matches ({100*matched/compared:.1f}%)")
    
    return df

## Load Data

In [ ]:
def load_deseq2(path):
    """Load DESeq2 output TSV (handles with/without header)."""
    if path is None:
        raise ValueError("No DESeq2 data provided.")
    
    # Check if file has header
    df = pd.read_csv(path, sep='\t', nrows=1)
    try:
        float(df.columns[1])
        # No header - first row is data
        df = pd.read_csv(path, sep='\t', header=None,
                        names=['Gene_ID', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj'])
    except ValueError:
        # Has header
        df = pd.read_csv(path, sep='\t')
        if df.columns[0] != 'Gene_ID':
            df = df.rename(columns={df.columns[0]: 'Gene_ID'})
    return df

def load_publication(path):
    """Load prepared publication CSV."""
    if path is None:
        raise ValueError("No publication data provided.")
    return pd.read_csv(path)

In [ ]:
# Helper to get path from Galaxy history or local file
async def get_path(history_id, local_file, name):
    """Get file path from Galaxy history ID or fall back to local file."""
    if history_id is not None:
        try:
            import gxy
            path = await gxy.get(history_id)
            print(f"{name}: Galaxy history #{history_id}")
            return path
        except ImportError:
            print(f"Warning: gxy not available, falling back to local file for {name}")
    
    if local_file is None:
        raise ValueError(f"No path configured for {name}")
    print(f"{name}: {local_file}")
    return local_file

# Load all files
print("=== Loading Files ===")
gtf_path = await get_path(GTF_V3_HISTORY_ID, GTF_V3_FILE, "GTF V3")
protein_v2_path = await get_path(PROTEIN_V2_HISTORY_ID, PROTEIN_V2_FILE, "Protein V2")
protein_v3_path = await get_path(PROTEIN_V3_HISTORY_ID, PROTEIN_V3_FILE, "Protein V3")
deseq2_path = await get_path(DESEQ2_HISTORY_ID, DESEQ2_FILE, "DESeq2")
publication_path = await get_path(PUBLICATION_HISTORY_ID, PUBLICATION_FILE, "Publication")

# Load DESeq2 and publication data
deseq2_df = load_deseq2(deseq2_path)
pub_df = load_publication(publication_path)

print(f"\nDESeq2 genes: {len(deseq2_df)}")
print(f"Publication genes: {len(pub_df)}")

## Build and Validate Gene Mapping

In [ ]:
# Parse GTF for gene ID mapping
v2_to_v3 = parse_gtf_gene_mapping(gtf_path)

# Parse protein sequences
v2_proteins = parse_fasta(protein_v2_path)
v3_proteins = parse_fasta(protein_v3_path)

# Filter mapping to genes in publication
pub_genes = set(pub_df['gene_id'])
relevant_mapping = {k: v for k, v in v2_to_v3.items() if k in pub_genes}
print(f"\nMapped {len(relevant_mapping)}/{len(pub_genes)} publication genes to V3 IDs")

# Validate with protein sequences
protein_validation = validate_mapping_with_proteins(relevant_mapping, v2_proteins, v3_proteins)

## Compare LFC Values

In [ ]:
def compare_lfc(pub_df, deseq2_df, v2_to_v3, key_genes=None):
    """
    Compare publication LFC to reanalysis LFC using official gene mapping.
    Automatically detects and corrects sign flip (comparison direction).
    """
    # Build DESeq2 lookup
    deseq2_lfc = dict(zip(deseq2_df['Gene_ID'], deseq2_df['log2FoldChange']))
    
    results = []
    for _, row in pub_df.iterrows():
        v2_id = row['gene_id']
        v3_id = v2_to_v3.get(v2_id)
        
        if v3_id and v3_id in deseq2_lfc:
            results.append({
                'pub_gene': v2_id,
                'our_gene': v3_id,
                'pub_lfc': row['log2fc'],
                'our_lfc': deseq2_lfc[v3_id],
                'gene_name': key_genes.get(v2_id, '') if key_genes else ''
            })
    
    df = pd.DataFrame(results)
    
    # Detect sign flip
    r_raw, _ = stats.pearsonr(df['pub_lfc'], df['our_lfc'])
    if r_raw < 0:
        df['our_lfc_corrected'] = -df['our_lfc']
        sign_flipped = True
    else:
        df['our_lfc_corrected'] = df['our_lfc']
        sign_flipped = False
    
    print(f"Compared {len(df)} genes")
    print(f"Sign flip detected: {sign_flipped}")
    
    return df, sign_flipped

# Run comparison
comparison_df, sign_flipped = compare_lfc(pub_df, deseq2_df, v2_to_v3, KEY_GENES)

In [ ]:
def calculate_metrics(df):
    """Calculate validation metrics."""
    pub = df['pub_lfc']
    our = df['our_lfc_corrected']
    
    r, _ = stats.pearsonr(pub, our)
    r2 = r ** 2
    slope, intercept = np.polyfit(pub, our, 1)
    direction_agree = np.mean(np.sign(pub) == np.sign(our)) * 100
    mean_diff = np.mean(np.abs(pub - our))
    
    return {
        'n_genes': len(df),
        'pearson_r2': r2,
        'slope': slope,
        'direction_agreement': direction_agree,
        'mean_abs_diff': mean_diff
    }

metrics = calculate_metrics(comparison_df)

print("\n=== Validation Metrics ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

## Visualization

In [ ]:
def plot_validation(df, name, metrics):
    """Generate interactive scatter plot with gene labels."""
    
    # Regression line data
    x_range = [df['pub_lfc'].min(), df['pub_lfc'].max()]
    slope, intercept = np.polyfit(df['pub_lfc'], df['our_lfc_corrected'], 1)
    reg_df = pd.DataFrame({
        'x': x_range,
        'y_reg': [slope * x + intercept for x in x_range],
        'y_identity': x_range
    })
    
    # Main scatter
    scatter = alt.Chart(df).mark_circle(size=60, opacity=0.7).encode(
        x=alt.X('pub_lfc:Q', title='Publication log2FC'),
        y=alt.Y('our_lfc_corrected:Q', title='Reanalysis log2FC'),
        tooltip=['pub_gene', 'our_gene', 'gene_name',
                 alt.Tooltip('pub_lfc:Q', format='.3f'),
                 alt.Tooltip('our_lfc_corrected:Q', format='.3f')]
    ).properties(
        title=f'{name} (R² = {metrics["pearson_r2"]:.4f})',
        width=500, height=500
    )
    
    # Gene labels for key genes
    labeled = df[df['gene_name'] != '']
    labels = alt.Chart(labeled).mark_text(
        align='left', dx=7, dy=-7, fontSize=12, fontWeight='bold', color='darkred'
    ).encode(
        x='pub_lfc:Q',
        y='our_lfc_corrected:Q',
        text='gene_name:N'
    )
    
    # Regression line
    reg_line = alt.Chart(reg_df).mark_line(color='red', strokeWidth=2).encode(
        x='x:Q', y='y_reg:Q'
    )
    
    # Identity line (y=x)
    identity = alt.Chart(reg_df).mark_line(
        color='black', strokeDash=[5, 5], opacity=0.5
    ).encode(x='x:Q', y='y_identity:Q')
    
    return scatter + labels + reg_line + identity

chart = plot_validation(comparison_df, COMPARISON_NAME, metrics)
chart

## Export Results

In [ ]:
# Save gene mapping
comparison_df.to_csv('gene_mapping.csv', index=False)
print("Saved gene_mapping.csv")

# Save protein validation
protein_validation.to_csv('protein_validation.csv', index=False)
print("Saved protein_validation.csv")

# Print summary
print(f"\n=== Summary ===")
print(f"Comparison: {COMPARISON_NAME}")
print(f"Genes mapped: {metrics['n_genes']}")
print(f"R² = {metrics['pearson_r2']:.4f}")
print(f"Direction agreement: {metrics['direction_agreement']:.1f}%")
print(f"Sign flip corrected: {sign_flipped}")